# 35 · Adapting the out-of-platform model to the target's reaction structure (E88)

Leave one platform out on TikTok / YouTube / X. Every adaptation uses only the target's inputs (likes, comments, shares, followers), never its views.

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import spearmanr

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "reach_samples_students.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_rows", 300)
import extra_models as em, history as H, platform_map as pm

In [2]:
f, meta = em.load(ROOT); W = pm.weight_summary(f, meta); W.round(3)

,target,effective sample share,weight share from youtube,weight share from twitter,mean mixture weight of youtube,weight share from tiktok,mean mixture weight of tiktok
0,tiktok,0.063,0.372,0.628,0.235,NaN,NaN
1,youtube,0.063,NaN,0.418,NaN,0.582,0.589
2,twitter,0.079,0.385,NaN,NaN,0.615,0.644


In [3]:
P = pm.adapt(f, meta); S = H.summarise(P); display(S.pivot(index="model", columns="platform", values="MAE")[pm.PLATS].round(3)); display(S.pivot(index="model", columns="platform", values="bias")[pm.PLATS].round(3))

platform,tiktok,youtube,twitter
model,,,
"CatBoost, importance weights",0.456,0.611,0.609
"CatBoost, raw",0.410,0.628,0.707
"CatBoost, raw + structure",0.482,0.639,0.814
"CatBoost, source mixture",0.445,0.623,0.728
"CatBoost, views per like",0.368,0.641,0.687
"CatBoost, views per reaction",0.364,0.626,0.662
"ridge, importance weights",0.385,0.602,0.618
"ridge, raw",0.312,0.604,0.641
"ridge, source mixture",0.476,0.631,0.626


platform,tiktok,youtube,twitter
model,,,
"CatBoost, importance weights",-0.208,-0.061,0.346
"CatBoost, raw",-0.176,-0.130,0.523
"CatBoost, raw + structure",-0.375,-0.093,0.716
"CatBoost, source mixture",-0.208,-0.108,0.537
"CatBoost, views per like",-0.038,-0.042,0.476
"CatBoost, views per reaction",-0.055,-0.057,0.443
"ridge, importance weights",0.083,-0.092,0.335
"ridge, raw",-0.053,-0.206,0.383
"ridge, source mixture",0.258,-0.189,0.358


In [4]:
G = pd.DataFrame([{"platform": p, "model": k, **dict(zip(["gain over ridge, raw", "CI low", "CI high"], H.gain_ci(P, p, "ridge, raw", k)))} for p in pm.PLATS for k in P.model.unique() if k != "ridge, raw"]); G.round(3)

,platform,model,"gain over ridge, raw",CI low,CI high
0,tiktok,"CatBoost, raw",-0.098,-0.112,-0.085
1,tiktok,"ridge, importance weights",-0.073,-0.082,-0.064
2,tiktok,"CatBoost, importance weights",-0.143,-0.159,-0.129
3,tiktok,"ridge, source mixture",-0.164,-0.181,-0.148
4,tiktok,"CatBoost, source mixture",-0.133,-0.151,-0.116
5,tiktok,"ridge, views per like",-0.002,-0.003,-0.002
6,tiktok,"CatBoost, views per like",-0.056,-0.065,-0.047
7,tiktok,"CatBoost, views per reaction",-0.051,-0.060,-0.042
8,tiktok,"CatBoost, raw + structure",-0.170,-0.186,-0.155
9,youtube,"CatBoost, raw",-0.023,-0.038,-0.008


In [5]:
R = P.groupby(["platform", "model"]).apply(lambda q: spearmanr(q.y, q.pred)[0]).unstack(0)[pm.PLATS]; R.round(3)

platform,tiktok,youtube,twitter
model,,,
"CatBoost, importance weights",0.854,0.685,0.551
"CatBoost, raw",0.862,0.679,0.541
"CatBoost, raw + structure",0.864,0.658,0.583
"CatBoost, source mixture",0.808,0.683,0.457
"CatBoost, views per like",0.870,0.649,0.516
"CatBoost, views per reaction",0.872,0.664,0.549
"ridge, importance weights",0.867,0.650,0.486
"ridge, raw",0.861,0.654,0.495
"ridge, source mixture",0.869,0.632,0.542


In [6]:
Wn = pm.with_shift(P); Wn.pivot_table(index="model", columns=["platform", "n"], values="MAE").round(3)

platform                     tiktok                      twitter                      youtube                     
n                                0      10     25     50      0      10     25     50      0      10     25     50
model                                                                                                             
CatBoost, importance weights  0.456  0.468  0.452  0.446   0.609  0.569  0.552  0.544   0.611  0.647  0.626  0.619
CatBoost, raw                 0.410  0.433  0.415  0.409   0.707  0.595  0.575  0.567   0.628  0.668  0.645  0.636
CatBoost, raw + structure     0.482  0.426  0.415  0.408   0.814  0.546  0.528  0.520   0.639  0.674  0.654  0.647
CatBoost, source mixture      0.445  0.458  0.445  0.439   0.728  0.614  0.593  0.585   0.623  0.663  0.641  0.633
CatBoost, views per like      0.368  0.390  0.380  0.374   0.687  0.598  0.577  0.569   0.641  0.687  0.659  0.648
CatBoost, views per reaction  0.364  0.386  0.375  0.369   0.662  0.583  0.562  0.555   0.626  0.666  0.644  0.635
ridge, importance weights     0.385  0.401  0.386  0.380   0.618  0.578  0.559  0.550   0.602  0.644  0.617  0.609
ridge, raw                    0.312  0.331  0.320  0.316   0.641  0.581  0.563  0.553   0.604  0.635  0.621  0.611
ridge, source mixture         0.476  0.453  0.438  0.431   0.626  0.577  0.555  0.547   0.631  0.668  0.647  0.639
ridge, views per like         0.315  0.333  0.322  0.318   0.629  0.577  0.557  0.551   0.608  0.645  0.623  0.615

In [7]:
json.dump({"weights": W.to_dict("records"), "models": S.to_dict("records"), "gains": G.to_dict("records"), "spearman": R.reset_index().to_dict("records"), "with_shift": Wn.to_dict("records")},
          open(ROOT / "data" / "processed" / "platform_adapt_numbers.json", "w"), indent=1, default=float)